In [ ]:
import os, glob, gc, json, copy, random, warnings
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchaudio, timm
from sklearn.metrics import average_precision_score
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

# Find competition data
for p in ['/kaggle/input/birdclef-2026', '/kaggle/input/competitions/birdclef-2026']:
    if os.path.exists(os.path.join(p, 'train.csv')):
        DATA = p; break
else:
    DATA = None
    for root, dirs, files in os.walk('/kaggle/input/'):
        if 'train.csv' in files and 'sample_submission.csv' in files:
            DATA = root; break
    assert DATA, 'Competition data not found!'
print(f'Data: {DATA}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SR = 32000; DUR = 5; AUDIO_LEN = SR * DUR; N_MELS = 128
EPOCHS = 12; BS = 32; LR = 3e-4
FOLDS_TO_TRAIN = [0, 1]
MODELS_DIR = '/kaggle/working/models/'
os.makedirs(MODELS_DIR, exist_ok=True)

sub_df = pd.read_csv(f'{DATA}/sample_submission.csv')
SPECIES = list(sub_df.columns)[1:]
NUM_CLASSES = len(SPECIES)
label2idx = {str(s): i for i, s in enumerate(SPECIES)}

df = pd.read_csv(f'{DATA}/train.csv')
df['primary_label'] = df['primary_label'].astype(str)
df['label_idx'] = df['primary_label'].map(label2idx)
df = df.dropna(subset=['label_idx']).copy()
df['label_idx'] = df['label_idx'].astype(int)
if 'rating' in df.columns:
    df = df[df['rating'].fillna(0) >= 1.5].reset_index(drop=True)
print(f'Species: {NUM_CLASSES}, Samples: {len(df)}')

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
df['fold'] = -1
for fold, (_, vi) in enumerate(skf.split(df, df['label_idx'])):
    df.loc[vi, 'fold'] = fold
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Capability: {torch.cuda.get_device_capability()}')
    print(f'PyTorch CUDA: {torch.version.cuda}')
    try:
        x = torch.randn(2, 3, 64, 64, device='cuda')
        y = F.pad(x, (1, 1, 1, 1))
        print(f'CUDA pad test: OK {y.shape}')
    except Exception as e:
        print(f'CUDA pad test FAILED: {e}')
        print('Falling back to CPU')
        DEVICE = torch.device('cpu')


In [ ]:
class BirdDataset(Dataset):
    def __init__(self, df, is_train=True):
        self.df = df
        self.is_train = is_train
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=SR, n_fft=2048, hop_length=512,
            n_mels=N_MELS, f_min=20, f_max=16000)
        self.db = torchaudio.transforms.AmplitudeToDB()
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fp = f"{DATA}/train_audio/{row['filename']}"
        try:
            info = torchaudio.info(fp)
            req = int(AUDIO_LEN * info.sample_rate / SR)
            if info.num_frames > req:
                off = random.randint(0, info.num_frames-req) if self.is_train else (info.num_frames-req)//2
                w, sr = torchaudio.load(fp, frame_offset=off, num_frames=req)
            else:
                w, sr = torchaudio.load(fp)
            if sr != SR: w = torchaudio.functional.resample(w, sr, SR)
            if w.shape[0] > 1: w = w.mean(0, keepdim=True)
            if w.shape[1] < AUDIO_LEN: w = F.pad(w, (0, AUDIO_LEN - w.shape[1]))
            else: w = w[:, :AUDIO_LEN]
        except Exception:
            w = torch.zeros((1, AUDIO_LEN))
        spec = self.db(self.mel(w))
        spec = spec.expand(3, -1, -1)
        spec = (spec - spec.mean()) / (spec.std() + 1e-6)
        label = torch.zeros(NUM_CLASSES)
        label[row['label_idx']] = 1.0
        return spec, label

print('Dataset defined.')


In [ ]:
class BirdModel(nn.Module):
    def __init__(self, backbone_name='tf_efficientnet_b0_ns'):
        super().__init__()
        self.backbone = timm.create_model(backbone_name, pretrained=True, num_classes=0, global_pool='avg')
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(self.backbone.num_features, NUM_CLASSES))
    def forward(self, x):
        return self.head(self.backbone(x))

def padded_cmap(y_true, y_pred, pad=5):
    aps = []
    for c in range(y_true.shape[1]):
        if y_true[:, c].sum() == 0: continue
        yt = np.concatenate([y_true[:, c], np.zeros(pad)])
        yp = np.concatenate([y_pred[:, c], np.zeros(pad)])
        aps.append(average_precision_score(yt, yp))
    return np.mean(aps) if aps else 0.0

print('Model defined.')


In [ ]:
def train_fold(fold, df, backbone='tf_efficientnet_b0_ns'):
    print(f"\n{'='*40} FOLD {fold} | {backbone} {'='*40}")
    random.seed(42+fold); np.random.seed(42+fold); torch.manual_seed(42+fold)
    train_df = df[df['fold'] != fold].reset_index(drop=True)
    valid_df = df[df['fold'] == fold].reset_index(drop=True)
    train_dl = DataLoader(BirdDataset(train_df, True), batch_size=BS,
                          shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
    valid_dl = DataLoader(BirdDataset(valid_df, False), batch_size=BS*2,
                          shuffle=False, num_workers=2, pin_memory=True)
    model = BirdModel(backbone).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, EPOCHS, 1e-6)
    criterion = nn.BCEWithLogitsLoss()
    best = -1
    for epoch in range(EPOCHS):
        model.train(); tloss = 0
        for imgs, labs in tqdm(train_dl, desc=f'E{epoch+1}', leave=False):
            imgs, labs = imgs.to(DEVICE), labs.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labs)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            tloss += loss.item()
        scheduler.step()
        model.eval(); ps, ts = [], []
        with torch.no_grad():
            for imgs, labs in valid_dl:
                logits = model(imgs.to(DEVICE))
                ps.append(torch.sigmoid(logits).cpu().numpy()); ts.append(labs.numpy())
        score = padded_cmap(np.vstack(ts), np.vstack(ps))
        print(f'E{epoch+1}/{EPOCHS} loss={tloss/len(train_dl):.4f} cMAP={score:.4f}')
        if score > best:
            best = score
            torch.save({'state_dict': model.state_dict(), 'backbone': backbone, 'val_cmap': score},
                       f'{MODELS_DIR}/best_{backbone}_fold{fold}.pth')
            print(f'  Saved! cMAP={score:.4f}')
    del model, optimizer, train_dl, valid_dl; gc.collect(); torch.cuda.empty_cache()

# Run training
for bb in ['tf_efficientnet_b0_ns', 'mobilenetv3_large_100']:
    for fold in FOLDS_TO_TRAIN:
        train_fold(fold, df, bb)
print('Training complete!')


In [ ]:
# Inference
mel_tf = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR, n_fft=2048, hop_length=512,
    n_mels=N_MELS, f_min=20, f_max=16000).to(DEVICE)
db_tf = torchaudio.transforms.AmplitudeToDB().to(DEVICE)

ensemble = []
for path in sorted(glob.glob(f'{MODELS_DIR}/best_*.pth')):
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    m = BirdModel(ckpt['backbone']).to(DEVICE)
    m.load_state_dict(ckpt['state_dict']); m.eval()
    ensemble.append((m, ckpt.get('val_cmap', 1.0)))
    print(f'Loaded {os.path.basename(path)} cMAP={ckpt.get("val_cmap",0):.4f}')
print(f'Ensemble: {len(ensemble)} models')
total_w = sum(a for _, a in ensemble)

test_files = sorted(glob.glob(f'{DATA}/test_soundscapes/*.ogg'))
if not test_files:
    print('No test soundscapes. Dummy submission.')
    sub = pd.read_csv(f'{DATA}/sample_submission.csv')
    sub.iloc[:, 1:] = 0.0
    sub.to_csv('submission.csv', index=False)
else:
    results = []
    for fp in tqdm(test_files, desc='Inference'):
        prefix = os.path.basename(fp).replace('.ogg', '')
        wt, sr = torchaudio.load(fp)
        if sr != SR: wt = torchaudio.functional.resample(wt, sr, SR)
        if wt.shape[0] > 1: wt = wt.mean(0, keepdim=True)
        wav = wt[0].numpy()
        n_bins = max(1, int(np.ceil(len(wav) / AUDIO_LEN)))
        for i in range(n_bins):
            s = i * AUDIO_LEN
            chunk = wav[s:s+AUDIO_LEN]
            if len(chunk) < AUDIO_LEN: chunk = np.pad(chunk, (0, AUDIO_LEN-len(chunk)))
            t = torch.tensor(chunk, device=DEVICE, dtype=torch.float32).unsqueeze(0).unsqueeze(0)
            spec = db_tf(mel_tf(t)).expand(1, 3, -1, -1)
            spec = (spec - spec.mean()) / (spec.std() + 1e-6)
            preds = np.zeros(NUM_CLASSES)
            with torch.no_grad():
                for mdl, auc in ensemble:
                    p = torch.sigmoid(mdl(spec)).cpu().numpy()[0]
                    preds += (auc / total_w) * p
            results.append([f'{prefix}_{(i+1)*5}'] + preds.tolist())
    sub = pd.DataFrame(results, columns=['row_id'] + SPECIES)
    sub.to_csv('submission.csv', index=False)
    print(f'submission.csv: {len(sub)} rows')
print('DONE!')
